# Equilibrium and transient optical response of the parabolic two-band model

Analysis of the parabolic two-band model to study the Dynamical Franz-Keldysh Effect (DFKE), adapted from the lecture notes of Shunsuke Sato.

The notebook reads time-dependent simulation output (vector potential, current, electric field) and computes:

1. the **equilibrium** optical conductivity $\sigma(\omega)$ from a probe-only run,
2. the **transient** conductivity $\tilde\sigma(\omega, T)$ at a single pump-probe delay,
3. the **transient absorption spectrum** $\Delta\sigma(\omega, T)$ over a scan of delays.

All heavy lifting is done by the tested `twoband` package in `src/twoband/`.

**Required data files** (place them in `data/`, see the README):

| File | Content |
|---|---|
| `Act_jt_3.6d8_probe.out` | probe only (equilibrium) |
| `Act_jt_3.6d10_pump.out` | pump only (reference) |
| `Act_jt_3d10_3d8_pp.out` | pump-probe at a single delay |
| `10_delays_3d10_3d8_absorption_spectra.dat` | pump-probe scan, one block of 50001 rows per delay |

Columns used: 0 = time (a.u.), 1 = A(t), 4 = J(t), 7 = E(t).

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

import twoband as tb

## Settings

In [ ]:
DATA_DIR = Path("data")

# Column layout of the simulation output
COL_TIME, COL_A, COL_J, COL_E = 0, 1, 4, 7

# Energy grid and damping, shared by ALL spectra so they can be compared directly
ENERGY = np.linspace(2.0, 20.0, 500) / tb.HARTREE_TO_EV  # Hartree
GAMMA = 0.25 / tb.HARTREE_TO_EV  # damping in the Fourier transform
ENERGY_EV = ENERGY * tb.HARTREE_TO_EV

# Delay scan layout
BLOCK_LENGTH = 50001  # rows per delay in the scan file
DELAY_START_FS, DELAY_END_FS = 1.0, 1.2  # check these against your simulation input


def load_trace(name):
    # Load one simulation output file from DATA_DIR with a clear error if it is missing.
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. See the Data section of the README.")
    return np.loadtxt(path)

## 1. Equilibrium optical conductivity

$$\sigma(\omega) = \frac{J(\omega)}{E(\omega)}, \qquad F(\omega) = \int F(t)\, e^{i\omega t - \gamma t}\, dt$$

In [ ]:
probe = load_trace("Act_jt_3.6d8_probe.out")
time = probe[:, COL_TIME]
probe_current = probe[:, COL_J]
probe_field = probe[:, COL_E]

fig, axs = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
axs[0].plot(time * tb.AU_TO_FS, probe[:, COL_A])
axs[0].set_ylabel("A(t) (a.u.)")
axs[1].plot(time * tb.AU_TO_FS, probe_current, color="r")
axs[1].set_ylabel("J(t) (a.u.)")
axs[2].plot(time * tb.AU_TO_FS, probe_field)
axs[2].set_ylabel(r"E$_{probe}$(t) (a.u.)")
axs[2].set_xlabel("Time (fs)")
plt.show()

In [ ]:
current_w = tb.damped_ft(time, probe_current, ENERGY, GAMMA)
field_w = tb.damped_ft(time, probe_field, ENERGY, GAMMA)
sigma_eq = current_w / field_w

fig, axs = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axs[0].plot(ENERGY_EV, np.abs(field_w), color="dodgerblue", label="FT probe field")
axs[0].set_ylabel("Intensity (arb. units)")
axs[0].legend()
axs[1].plot(ENERGY_EV, np.abs(current_w), color="red", label="FT current")
axs[1].set_ylabel("Intensity (arb. units)")
axs[1].set_xlabel("Energy (eV)")
axs[1].legend()
plt.show()

plt.plot(ENERGY_EV, -np.real(sigma_eq), color="red", linewidth=2, label="Equilibrium")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\sigma(\omega)$] (arb. units)")
plt.legend()
plt.show()

## 2. Pump-only reference

In [ ]:
pump = load_trace("Act_jt_3.6d10_pump.out")
pump_time = pump[:, COL_TIME]
pump_current = pump[:, COL_J]
pump_field = pump[:, COL_E]

fig, axs = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
axs[0].plot(pump_time * tb.AU_TO_FS, pump[:, COL_A])
axs[0].set_ylabel("A(t) (a.u.)")
axs[1].plot(pump_time * tb.AU_TO_FS, pump_current, color="r")
axs[1].set_ylabel(r"J$_{pump}$(t) (a.u.)")
axs[2].plot(pump_time * tb.AU_TO_FS, pump_field)
axs[2].set_ylabel(r"E$_{pump}$(t) (a.u.)")
axs[2].set_xlabel("Time (fs)")
plt.show()

## 3. Transient conductivity at a single delay

The probe response is isolated by subtracting the pump-only run:

$$J_{probe}(t) = J_{pump\text{-}probe}(t) - J_{pump}(t), \qquad E_{probe}(t) = E_{pump\text{-}probe}(t) - E_{pump}(t)$$

Both spectra use the same `ENERGY` grid as the equilibrium result, so they can be subtracted directly.

In [ ]:
pp = load_trace("Act_jt_3d10_3d8_pp.out")
pp_time = pp[:, COL_TIME]
pp_probe_current = tb.subtract_reference(pp[:, COL_J], pump_current)
pp_probe_field = tb.subtract_reference(pp[:, COL_E], pump_field)

fig, axs = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axs[0].plot(pp_time * tb.AU_TO_FS, pp_probe_field)
axs[0].set_ylabel(r"E$_{probe}$(t) (a.u.)")
axs[1].plot(pp_time * tb.AU_TO_FS, pp_probe_current, color="r")
axs[1].set_ylabel(r"J$_{probe}$(t) (a.u.)")
axs[1].set_xlabel("Time (fs)")
plt.show()

In [ ]:
sigma_single = tb.optical_conductivity(pp_time, pp_probe_current, pp_probe_field, ENERGY, GAMMA)
delta_single = tb.transient_absorption(sigma_single, sigma_eq)[0]

plt.figure(figsize=(8, 6))
plt.plot(ENERGY_EV, -np.real(sigma_single), label="Transient")
plt.plot(ENERGY_EV, -np.real(sigma_eq), label="Equilibrium")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\sigma(\omega)$] (arb. units)")
plt.legend()
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(ENERGY_EV, -np.real(delta_single), label="Difference")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\sigma(\omega,T)-\sigma(\omega)$] (arb. units)")
plt.legend()
plt.show()

## 4. Transient absorption spectrum over a delay scan

The scan file contains one block of `BLOCK_LENGTH` rows per delay. `split_delays` refuses to split data that does not divide evenly, so a truncated file is caught instead of silently producing a wrong map.

$$\Delta\sigma(\omega, T) = \tilde\sigma(\omega, T) - \sigma(\omega)$$

In [ ]:
scan = load_trace("10_delays_3d10_3d8_absorption_spectra.dat")
blocks = tb.split_delays(scan, BLOCK_LENGTH)  # shape (n_delays, BLOCK_LENGTH, n_columns)
n_delays = blocks.shape[0]
scan_time = blocks[0, :, COL_TIME]
print(f"{n_delays} delays, {BLOCK_LENGTH} time steps each")

scan_probe_current = tb.subtract_reference(blocks[:, :, COL_J], pump_current)
scan_probe_field = tb.subtract_reference(blocks[:, :, COL_E], pump_field)

fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for i in range(n_delays):
    axs[0].plot(scan_time * tb.AU_TO_FS, scan_probe_field[i])
    axs[1].plot(scan_time * tb.AU_TO_FS, scan_probe_current[i])
axs[0].set_ylabel(r"E$_{probe}$(t)")
axs[1].set_ylabel(r"J$_{probe}$(t)")
axs[1].set_xlabel("Time (fs)")
plt.show()

In [ ]:
sigma_scan = tb.optical_conductivity(scan_time, scan_probe_current, scan_probe_field, ENERGY, GAMMA)
tas = tb.transient_absorption(sigma_scan, sigma_eq)  # shape (n_delays, n_energy)

plt.plot(ENERGY_EV, -np.real(sigma_scan).T, linewidth=1)
plt.plot(ENERGY_EV, -np.real(sigma_eq), color="k", linewidth=2, label="Equilibrium")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\sigma(\omega)$] (arb. units)")
plt.legend()
plt.show()

plt.plot(ENERGY_EV, -np.real(tas).T)
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\Delta\sigma(\omega,T)$] (arb. units)")
plt.show()

### Transient absorption map

In [ ]:
delays_fs = np.linspace(DELAY_START_FS, DELAY_END_FS, n_delays)
limit = 0.005

fig, ax = plt.subplots(figsize=(5, 4))
mesh = ax.pcolormesh(
    delays_fs,
    ENERGY_EV,
    -np.real(tas).T,
    cmap="seismic",
    vmin=-limit,
    vmax=limit,
    shading="nearest",
)
fig.colorbar(mesh, ax=ax, label=r"$-$Re[$\Delta\sigma$]")
ax.set_ylim(3, 8)
ax.set_xlabel("Time delay (fs)")
ax.set_ylabel("Energy (eV)")
plt.show()

## Reference

S. A. Sato et al., *Applied Sciences* 8(10), 1777 (2018). https://www.mdpi.com/2076-3417/8/10/1777